# NB03 — Baseline LSTM et Entraînement
> **Phase C & D** : Hochreiter & Schmidhuber 1997, Bengio et al. 2003.


In [1]:
# ==============================================================================
# Cellule 1 : Paramètres (Settings) et reproductibilité
# ==============================================================================
import sys
import math
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from utils import set_seed, get_device, AminoAcidTokenizer, SequenceDataset, collate_fn, compute_nats_and_perplexity

SEED = 42
set_seed(SEED)
device = get_device()

BATCH_SIZE = 64
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
LR = 1e-3
WEIGHT_DECAY = 0.01
EPOCHS = 10
GRAD_CLIP = 1.0

print(f"Device: {device} | Seed: {SEED} | Epochs: {EPOCHS} | LR: {LR}")


C:\Users\USER\AppData\Local\Temp\ipykernel_18436\1170114651.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


ModuleNotFoundError: No module named 'torch'

In [ ]:
# ==============================================================================
# Cellule 2 : Architecture LSTM avec Weight Tying
# ==============================================================================
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.proj = nn.Linear(hidden_dim, embed_dim) if hidden_dim != embed_dim else nn.Identity()
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        # Weight tying : head.weight = embed.weight
        self.head.weight = self.embed.weight

    def forward(self, x, hidden=None):
        emb = self.embed(x)
        out, hidden = self.lstm(emb, hidden)
        out = self.proj(out)
        logits = self.head(out)
        return logits, hidden

tokenizer = AminoAcidTokenizer()
model = LSTMLM(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modèle LSTM instancié : {n_params:,} paramètres entraînables.")


In [ ]:
# ==============================================================================
# Cellule 3 : Données, Optimiseur AdamW et Fonction d'évaluation (nats/token)
# ==============================================================================
train_path = ROOT_DIR / "data" / "train.csv"
val_path = ROOT_DIR / "data" / "val.csv"
if not train_path.exists():
    train_path = ROOT_DIR / "data" / "toy" / "train.csv"
    val_path = ROOT_DIR / "data" / "toy" / "val.csv"

train_seqs = pd.read_csv(train_path)["sequence"].tolist()
val_seqs = pd.read_csv(val_path)["sequence"].tolist()

train_loader = DataLoader(
    SequenceDataset(train_seqs, tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_id)
)
val_loader = DataLoader(
    SequenceDataset(val_seqs, tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id=tokenizer.pad_id)
)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id, reduction='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
            non_pad = (targets != tokenizer.pad_id).sum().item()
            total_loss += loss.item()
            total_tokens += non_pad
    return compute_nats_and_perplexity(total_loss, total_tokens)

print(f"Données prêtes : Train={len(train_seqs):,} | Val={len(val_seqs):,}")


In [ ]:
# ==============================================================================
# Cellule 4 : Boucle d'entraînement (10 epochs) et sauvegarde du meilleur modèle
# ==============================================================================
import matplotlib.pyplot as plt

history = []
best_val_nats = float('inf')
models_dir = ROOT_DIR / "models"
models_dir.mkdir(exist_ok=True)
best_model_path = models_dir / "lstm_best.pt"

print("Début de l'entraînement LSTM...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    total_train_tokens = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
        non_pad = (targets != tokenizer.pad_id).sum().item()

        (loss / max(non_pad, 1)).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_train_loss += loss.item()
        total_train_tokens += non_pad

    train_nats, train_ppl = compute_nats_and_perplexity(total_train_loss, total_train_tokens)
    val_nats, val_ppl = evaluate(model, val_loader)
    history.append((epoch, train_nats, val_nats))

    if val_nats < best_val_nats:
        best_val_nats = val_nats
        torch.save(model.state_dict(), best_model_path)
        star = "★"
    else:
        star = ""

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train nats: {train_nats:.3f} (ppl: {train_ppl:.1f}) | Val nats: {val_nats:.3f} (ppl: {val_ppl:.1f}) {star}")

print(f"Entraînement terminé. Meilleur val nats: {best_val_nats:.3f} sauvegardé dans {best_model_path}")

# Affichage des courbes
epochs_range = [h[0] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, [h[1] for h in history], label="Train nats/token")
plt.plot(epochs_range, [h[2] for h in history], label="Val nats/token")
plt.xlabel("Epoch")
plt.ylabel("nats / token")
plt.title("Courbe d'apprentissage LSTM")
plt.legend()
plt.tight_layout()
plt.show()
